# Surrogate Factory — UCCpHTP
## Chapter 2. Data Acquisition
Objectives:
- Load the pre-split HTP CFD dataset.
- Files expected in `UCCpHTP/data/`: `x_train.csv`, `x_val.csv`, `x_test.csv`, `yt_train.csv`, `yt_val.csv`, `yt_test.csv`.
- Merge x + yt to build Train / Val / Test sets.

### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
import pandas as pd
from pathlib import Path
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

### 2. Data Acquisition — Load pre-split files

In [ ]:
workflow.import_metadata(stage_name="SF_2_Data_Acquisition_Generation")

In [ ]:
import re

split_dir = Path(workflow.config['data_split_folder'])

# Names the rest of the pipeline uses (SF_5 / SF_6 metadata). The pre-split CSVs
# come straight from the CFD export, so neither their headers nor their layout
# are guaranteed: observed in practice are ';' separators and files with no
# header row at all, where the first data line would otherwise be read as the
# column names.
INPUTS  = ['x', 'y', 'z', 'alpha', 'mach']
OUTPUTS = ['Cp']


def _sniff(path):
    """Work out the delimiter, and whether the first line is a header."""
    with open(path) as fh:
        first = fh.readline().rstrip('\r\n')

    counts = {s: first.count(s) for s in (';', ',', '\t', '|')}
    sep = max(counts, key=counts.get)
    if counts[sep] == 0:
        sep = ','                       # single column: any separator will do

    def numeric(tok):
        try:
            float(tok.strip().replace(',', '.') if sep != ',' else tok.strip())
            return True
        except ValueError:
            return False

    fields = [f for f in first.split(sep) if f.strip() != '']
    # All-numeric first line means the export wrote no header.
    header = None if fields and all(numeric(f) for f in fields) else 0
    return sep, header


def load_split(filename, expected, what):
    """Read a split CSV and return it with the pipeline's column names."""
    path = split_dir / filename
    sep, header = _sniff(path)
    df = pd.read_csv(path, sep=sep, header=header)
    if header is None:
        print(f"  {filename}: no header row, sep={sep!r} -> naming columns {expected}")
    elif sep != ',':
        print(f"  {filename}: sep={sep!r}")

    junk = [c for c in df.columns if re.fullmatch(r'Unnamed: \d+', str(c))]
    if junk:
        df = df.drop(columns=junk)
        print(f"  {filename}: dropped index column(s) {junk}")

    # With no header an index column has no name to recognise it by, so spot it
    # by shape: one column too many, the first being a consecutive integer run.
    if header is None and len(df.columns) == len(expected) + 1:
        first = df.iloc[:, 0]
        step = first.diff().dropna()
        if (first % 1 == 0).all() and (step == 1).all():
            df = df.iloc[:, 1:]
            print(f"  {filename}: dropped unnamed positional index column")

    if len(df.columns) != len(expected):
        raise ValueError(
            f"{filename}: expected {len(expected)} {what} column(s) {expected}, "
            f"but the file has {len(df.columns)}: {list(df.columns)[:8]}.\n"
            f"Detected separator {sep!r}, header={'yes' if header == 0 else 'no'}. "
            f"Check the file, or edit INPUTS/OUTPUTS to match your export."
        )

    if list(df.columns) != expected:
        if header is None:
            # No names in the file: position is the only information there is.
            df = df.set_axis(expected, axis=1)
        else:
            # The file does name its columns, so match on the name rather than
            # the position — renaming positionally would silently mislabel a
            # file whose columns are in a different order.
            norm = {str(c).strip().lower().lstrip('﻿'): c for c in df.columns}
            matched = {e: norm.get(e.lower()) for e in expected}
            if all(v is not None for v in matched.values()):
                df = df[[matched[e] for e in expected]].set_axis(expected, axis=1)
                print(f"  {filename}: matched columns by name -> {expected}")
            else:
                unmatched = [e for e, v in matched.items() if v is None]
                print(f"  {filename}: WARNING no column named {unmatched} in "
                      f"{list(df.columns)}; falling back to position "
                      f"{list(df.columns)} -> {expected}. Check the order is right.")
                df = df.set_axis(expected, axis=1)

    return df.apply(pd.to_numeric, errors='coerce')


x_train = load_split('x_train.csv', INPUTS, 'input')
x_val   = load_split('x_val.csv',   INPUTS, 'input')
x_test  = load_split('x_test.csv',  INPUTS, 'input')

yt_train = load_split('yt_train.csv', OUTPUTS, 'output')
yt_val   = load_split('yt_val.csv',   OUTPUTS, 'output')
yt_test  = load_split('yt_test.csv',  OUTPUTS, 'output')

Train_set = pd.concat([x_train.reset_index(drop=True), yt_train.reset_index(drop=True)], axis=1)
Val_set   = pd.concat([x_val.reset_index(drop=True),   yt_val.reset_index(drop=True)],   axis=1)
Test_set  = pd.concat([x_test.reset_index(drop=True),  yt_test.reset_index(drop=True)],  axis=1)

for name, df in (('Train', Train_set), ('Val', Val_set), ('Test', Test_set)):
    missing = [c for c in INPUTS + OUTPUTS if c not in df.columns]
    if missing:
        raise ValueError(f"{name}_set is missing {missing} — got {list(df.columns)}")
    nulls = int(df.isna().sum().sum())
    if nulls:
        print(f"  WARNING: {name}_set has {nulls:,} non-numeric/missing cell(s)")

print(f"\nTrain : {Train_set.shape[0]:>8,} rows   columns: {list(Train_set.columns)}")
print(f"Val   : {Val_set.shape[0]:>8,} rows")
print(f"Test  : {Test_set.shape[0]:>8,} rows")
Train_set.describe()


### Save

In [ ]:
job = workflow.config['job_name']
workflow.save_data(Train_set, job + '_Train_set.csv')
workflow.save_data(Val_set,   job + '_Val_set.csv')
workflow.save_data(Test_set,  job + '_Test_set.csv')
workflow.save_metadata()